# 05 — Summarization & One-Page Briefs

**Goal:** generate a short extractive summary of each company's MD&A and Risk Factors sections
using a pre-trained Hugging Face model, and combine everything from notebooks 01-04 into a
one-page brief per company.

Recall from Phase 2: we use **extractive** summarization as primary (pulls real sentences from the
text, never invents anything) rather than abstractive (which can hallucinate numbers/claims) —
important for financial content where invented facts are a real risk, not just a style choice.

## Setup — install if needed

In [1]:
import os

PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\Devanshi\financial-nlp-analyzer


In [2]:
# Run once if not already installed:
# pip install transformers torch --break-system-packages

import os
import pandas as pd
from transformers import pipeline

EXTRACTED_DIR = "data/extracted_text"
PROCESSED_DIR = "data/processed"
BRIEFS_DIR = "outputs/one_page_briefs"
os.makedirs(BRIEFS_DIR, exist_ok=True)

COMPANIES = ["jpmorgan", "caterpillar", "accenture", "pg", "infosys"]


c:\Users\Devanshi\financial-nlp-analyzer\financial_analyzer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load a pre-trained summarization model

Using a smaller distilled model (`sshleifer/distilbart-cnn-12-6`) rather than a full-size one —
noticeably faster on CPU with a small quality trade-off, which is the right call for this project
(see Phase 4's compute notes).

In [3]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.12.1+cpu
CUDA available: False


In [7]:
import torch
print(torch.__version__)

2.12.1+cpu


In [5]:
import sys
print(sys.executable)

c:\Users\Devanshi\financial-nlp-analyzer\financial_analyzer\Scripts\python.exe


In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarize_text(text, max_length=120, min_length=30):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    summary_ids = model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        do_sample=False,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 27946.93it/s]


In [5]:
import transformers
print(transformers.__version__)

from transformers.pipelines import PIPELINE_REGISTRY
print("summarization" in PIPELINE_REGISTRY.supported_tasks)

5.13.0
False


## Load extracted text

In [6]:
def load_section(company, section):
    path = os.path.join(EXTRACTED_DIR, f"{company}_{section}.txt")
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

texts = {company: {"mdna": load_section(company, "mdna"), "risk": load_section(company, "risk")}
         for company in COMPANIES}


## Summarization function

Hugging Face summarization models have an input length limit (usually ~1024 tokens). Long MD&A
sections need to be chunked first, summarized chunk-by-chunk, then optionally summarized again
('summarize the summaries') for a single short output.

In [7]:
def chunk_text(text, max_chars=3000):
    """Splits text into rough chunks that fit the model's input limit."""
    words = text.split()
    chunks, current = [], []
    current_len = 0
    for w in words:
        current.append(w)
        current_len += len(w) + 1
        if current_len >= max_chars:
            chunks.append(" ".join(current))
            current, current_len = [], 0
    if current:
        chunks.append(" ".join(current))
    return chunks


def summarize_long_text(text, max_length=120, min_length=30):
    """Chunks long text, summarizes each chunk, then summarizes the combined summaries."""
    if text is None or len(text.strip()) == 0:
        return None

    chunks = chunk_text(text)
    chunk_summaries = []
    for chunk in chunks:
        if len(chunk.split()) < 20:  # skip trivially short chunks
            continue
        summary = summarize_text(chunk, max_length=max_length, min_length=min_length)
        chunk_summaries.append(summary)

    combined = " ".join(chunk_summaries)

    if len(chunk_summaries) > 1:
        return summarize_text(combined, max_length=max_length, min_length=min_length)
    return combined


## Generate summaries for every company (this may take a few minutes on CPU)

In [8]:
summaries = {}

for company in COMPANIES:
    print(f"Summarizing {company}...")
    mdna_summary = summarize_long_text(texts[company]["mdna"])
    risk_summary = summarize_long_text(texts[company]["risk"])
    summaries[company] = {"mdna_summary": mdna_summary, "risk_summary": risk_summary}
    print(f"  MD&A summary: {mdna_summary[:150] if mdna_summary else 'N/A'}...")
    print(f"  Risk summary: {risk_summary[:150] if risk_summary else 'N/A'}...")
    print()


Summarizing jpmorgan...
  MD&A summary:  Management’s discussion and analysis of financial condition and results of operations appears on pages 52–167 . Such information should be read in co...
  Risk summary:  Any of the risk factors discussed below could by itself, or combined with other factors, materially and adversely affect JPMorganChase’s business, re...

Summarizing caterpillar...
  MD&A summary:  Cat Financial provides retail and wholesale financing alternatives to customers and dealers for Caterpillar products and services . Cat Financial fin...
  Risk summary:  The information contained on the company’s website is not included in, or incorporated by reference into, this annual report on Form 10-K . The occur...

Summarizing accenture...
  MD&A summary:  Revenues of $64.9 billion, an increase of 1% in U.S. dollars and 2% in local currency . Operating margin of 14.8%, compared to 13.7% in fiscal 2023 ....
  Risk summary:  Risks in this section are grouped in the following cate

## Build the final one-page brief per company (combines all prior notebooks' outputs)

In [9]:
extractions_path = os.path.join(PROCESSED_DIR, "company_extractions.csv")
comparison_path = os.path.join(PROCESSED_DIR, "cross_company_comparison.csv")

extractions_df = pd.read_csv(extractions_path) if os.path.exists(extractions_path) else pd.DataFrame()
comparison_df = pd.read_csv(comparison_path) if os.path.exists(comparison_path) else pd.DataFrame()

def build_brief(company):
    lines = [f"ONE-PAGE BRIEF: {company.upper()}", "=" * 50, ""]

    lines.append("MD&A SUMMARY:")
    lines.append(summaries[company]["mdna_summary"] or "N/A")
    lines.append("")

    lines.append("RISK FACTORS SUMMARY:")
    lines.append(summaries[company]["risk_summary"] or "N/A")
    lines.append("")

    if not comparison_df.empty and company in comparison_df["company"].values:
        row = comparison_df[comparison_df["company"] == company].iloc[0]
        lines.append("KEY METRICS:")
        for col in comparison_df.columns:
            if col != "company" and pd.notna(row[col]):
                lines.append(f"  {col}: {row[col]}")

    return "\n".join(lines)


for company in COMPANIES:
    brief_text = build_brief(company)
    out_path = os.path.join(BRIEFS_DIR, f"{company}_brief.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(brief_text)
    print(f"Saved {out_path}")


Saved outputs/one_page_briefs\jpmorgan_brief.txt
Saved outputs/one_page_briefs\caterpillar_brief.txt
Saved outputs/one_page_briefs\accenture_brief.txt
Saved outputs/one_page_briefs\pg_brief.txt
Saved outputs/one_page_briefs\infosys_brief.txt


## Next step

These `.txt` briefs are your raw one-page analyst summaries per company. From here, you can:
- Format them nicely into a PDF (see the `pdf` skill / Phase 4 tooling)
- Feed `cross_company_comparison.csv` into Power BI for the cross-sector dashboard (Phase 9)

In [11]:
from fpdf import FPDF
import os

BRIEFS_DIR = "outputs/one_page_briefs"
PDF_DIR = "outputs/one_page_briefs_pdf"
os.makedirs(PDF_DIR, exist_ok=True)

def sanitize(text):
    """Replace common Unicode punctuation with Latin-1-safe equivalents."""
    replacements = {
        "\u2014": "-",   # em dash —
        "\u2013": "-",   # en dash –
        "\u2018": "'",   # left single quote '
        "\u2019": "'",   # right single quote '
        "\u201c": '"',   # left double quote "
        "\u201d": '"',   # right double quote "
        "\u2022": "-",   # bullet •
        "\u2026": "...", # ellipsis …
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text.encode("latin-1", errors="replace").decode("latin-1")

def txt_brief_to_pdf(company):
    with open(os.path.join(BRIEFS_DIR, f"{company}_brief.txt"), "r", encoding="utf-8") as f:
        lines = f.read().split("\n")

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, sanitize(f"{company.upper()} - One-Page Analyst Brief"), ln=True)
    pdf.set_draw_color(30, 30, 30)
    pdf.line(10, 20, 200, 20)
    pdf.ln(5)

    for line in lines:
        clean_line = sanitize(line)
        if clean_line.isupper() and clean_line.strip().endswith(":"):
            pdf.set_font("Helvetica", "B", 12)
            pdf.ln(3)
            pdf.multi_cell(0, 7, clean_line)
            pdf.set_font("Helvetica", "", 10)
        else:
            pdf.multi_cell(0, 6, clean_line)

    pdf.output(os.path.join(PDF_DIR, f"{company}_brief.pdf"))
    print(f"Saved {company}_brief.pdf")

COMPANIES = ["jpmorgan", "caterpillar", "accenture", "pg", "infosys"]
for c in COMPANIES:
    txt_brief_to_pdf(c)

Saved jpmorgan_brief.pdf
Saved caterpillar_brief.pdf
Saved accenture_brief.pdf
Saved pg_brief.pdf
Saved infosys_brief.pdf
